<a href="https://colab.research.google.com/github/JAYAPRIYA9601/MINI-PROJECT/blob/main/Spine_segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



> The results obtained from the model was different during every execution.`But the accuracy seems to be consistent (around 85% -95%).` Try increasing the number of images. The epochs for whuch the model was trained till now was:5,20 but increasing the steps from 50 to 75.









In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%tensorflow_version 1.x

In [ ]:
import tensorflow as tf

#from data import *

from keras.models import *
from keras.preprocessing.image import ImageDataGenerator
from keras.layers import *
from keras.optimizers import *
import numpy as np 
import os
import skimage.io as io
import skimage.transform as trans
import numpy as np
from keras import backend as keras
from keras.callbacks import ModelCheckpoint, LearningRateScheduler, EarlyStopping, ReduceLROnPlateau#
import matplotlib.pyplot as plt
import pickle


In [ ]:
#data augumentation 
data_gen_args = dict(featurewise_center=False,
                     featurewise_std_normalization=False,
                     width_shift_range=0.1,
                     height_shift_range=0.1,
                     zoom_range=0.2)
image_datagen = ImageDataGenerator(**data_gen_args,rescale=1.0/255.0)   #normaliztion for original image
seed = 1
mask_datagen = ImageDataGenerator(**data_gen_args,rescale=1.0/255.0)     #normalization for mask image 

In [ ]:
tf.compat.v1.global_variables# to initialise the global varibles of tensorflow 1

In [ ]:
#generator = image_datagen.flow_from_directory('/content/drive/My Drive/original',target_size=(256,256),save_to_dir='/content/drive/My Drive/mask',class_mode=None,save_prefix='N',save_format='png',color_mode="grayscale")

In [ ]:
#original image generator 
image_generator = image_datagen.flow_from_directory(
             '/content/drive/My Drive/Original_GE',
             target_size=(256,256),
             seed=seed,
             class_mode=None,
            color_mode='grayscale')
    

In [ ]:
#mask image generator 
mask_generator = mask_datagen.flow_from_directory(
        '/content/drive/My Drive/Mask_GE',
        target_size=(256,256),
        class_mode=None,
        seed=seed,
        color_mode='grayscale')

In [ ]:
validation_mask_generator = mask_datagen.flow_from_directory(
        '/content/drive/My Drive/validation_mask',
        target_size=(256,256),
        class_mode=None,
        seed=seed,
        color_mode='grayscale')

In [ ]:
validation_image_generator = image_datagen.flow_from_directory(
             '/content/drive/My Drive/validation_original',
             target_size=(256,256),
             seed=seed,
             class_mode=None,
            color_mode='grayscale')

In [ ]:
def my_image_mask_generator(image_data_generator, mask_data_generator):

  validation_generator = zip(validation_image_generator, validation_mask_generator)
  for (img, mask) in validation_generator:
        yield (img, mask)  

val_generator = my_image_mask_generator(validation_image_generator,validation_mask_generator)


In [ ]:
  #combining the original and mask into another new generator to feed it into the model
def my_image_mask_generator(image_data_generator, mask_data_generator):
    train_generator = zip(image_generator, mask_generator)
    for (img, mask) in train_generator:
          yield (img, mask)  

mygen = my_image_mask_generator(image_generator,mask_generator)


In [ ]:
#for the purpose of testing 
test_generator = image_datagen.flow_from_directory(
             '/content/drive/My Drive/Test folder',
             target_size=(256,256),
             seed=seed,
            color_mode='grayscale')

In [ ]:
smooth=1
def dice_coef(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)


def dice_coef_loss(y_true, y_pred):
    return -dice_coef(y_true, y_pred)


In [ ]:
def unet(pretrained_weights = None,input_size = (256,256,1)):
    inputs = Input(input_size)
    conv1 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(inputs)
    conv1 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv1)
    pool1 = MaxPool2D(pool_size=(2, 2))(conv1)
    conv2 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool1)
    conv2 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv2)
    pool2 = MaxPool2D(pool_size=(2, 2))(conv2)
    conv3 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool2)
    conv3 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv3)
    pool3 = MaxPool2D(pool_size=(2, 2))(conv3)
    conv4 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool3)
    conv4 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv4)
    drop4 = Dropout(0.5)(conv4)
    pool4 = MaxPool2D(pool_size=(2, 2))(drop4)
    conv5 = Conv2D(1024, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(pool4)
    conv5 = Conv2D(1024, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv5)
    drop5 = Dropout(0.5)(conv5)
    up6 = Conv2D(512, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(drop5))
    merge6 = concatenate([drop4,up6], axis = 3)
    conv6 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge6)
    conv6 = Conv2D(512, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv6)
    up7 = Conv2D(256, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv6))
    merge7 = concatenate([conv3,up7], axis = 3)
    conv7 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge7)
    conv7 = Conv2D(256, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv7)
    up8 = Conv2D(128, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv7))
    merge8 = concatenate([conv2,up8], axis = 3)
    conv8 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge8)
    conv8 = Conv2D(128, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv8)

    up9 = Conv2D(64, 2, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(UpSampling2D(size = (2,2))(conv8))
    merge9 = concatenate([conv1,up9], axis = 3)
    conv9 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(merge9)
    conv9 = Conv2D(64, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv9)
    conv9 = Conv2D(2, 3, activation = 'relu', padding = 'same', kernel_initializer = 'he_normal')(conv9)
    conv10 = Conv2D(1, 1, activation = 'sigmoid')(conv9)
    model = Model(input = inputs, output = conv10)

    model.compile(optimizer = Adam(lr=1e-4), loss =dice_coef_loss, metrics = ['accuracy',dice_coef])
    
    #model.summary()

    if(pretrained_weights):
    	model.load_weights(pretrained_weights)

    return model



In [ ]:
model = unet()

In [ ]:
model.summary()

In [ ]:
history=model.fit_generator(mygen,steps_per_epoch=75,epochs=20,verbose=1,validation_data=val_generator,validation_steps=10)  

In [ ]:
#MODEL PERFORMANCE
import matplotlib.pyplot as plt
import numpy as np
#  "Accuracy"
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'validation'], loc='upper left')
plt.show()
    

In [ ]:
y_pred=model.predict_generator(test_generator,verbose=1) 
y_pred.shape
imgs=np.resize(y_pred,(256,256))
plt.imshow(imgs,cmap='gray')




In [ ]:
y_pred=model.predict(test_generator,verbose=1) 
y_pred.shape
imgs=np.resize(y_pred,(256,256))
plt.imshow(imgs,cmap='gray')

In [ ]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

In [ ]:
#to get a test image of size (128,128,1)
import cv2
img=cv2.imread('/content/drive/My Drive/projectdataset/Original_GE/original (1)/02(orig).png')
gray_image = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
width = 128
height = 128
dim = (width, height)
 
# resize image
resized = cv2.resize(gray_image, dim, interpolation = cv2.INTER_AREA) 
y = np.expand_dims(resized, axis=-1)
print(y.shape)
#resized.shape
cv2.imwrite('/content/drive/My Drive/projectdataset/Original_GE/Trial03.png',y)


In [ ]:
hell=cv2.imread('/content/drive/My Drive/projectdataset/Original_GE/trial.png')
hell.shape

In [ ]:
trial=cv2.imread('/content/drive/My Drive/projectdataset/Original_GE/02(orig)trial.png')
trial.shape

In [ ]:
tf.__version__

In [ ]:
import os

In [ ]:
model.save('/content/drive/My Drive/projectdataset/spine_model_validation.h5')

In [ ]:
model.save('my_model_validation.h5')

In [ ]:
from __future__ import absolute_import, division, print_function, unicode_literals
#tf.enable_v2_behavior()

## Cells below this are for experimental purpose


In [ ]:
%tensorflow_version 2.x

In [ ]:
model.fit(validation_split)

In [ ]:
y_pred =model.predict("\")
plt.imshow(y_pred,cmap='grayscale')

In [ ]:
# Import PyDrive and associated libraries.
# This only needs to be done once in a notebook.
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

# Authenticate and create the PyDrive client.
# This only needs to be done once in a notebook.
auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)



In [ ]:
tf.global_variables_initializer